In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("GLD VS GC=F RollingWindow = 180.csv").rename(columns={"index":"date"})
data.head()

,date,GLD_Close,GLD_dailyReturns,GC=F_Close,GC=F_dailyReturns,spread,abs_sprd,GLD_rolling avg,GC=F_rolling avg,isSprdPos,...,GLD_PnL,GC=F_PnL,Total PnL,riskThreshold,grossCashflow,totalMTM,totalMTM in %,isStoppedOut,Adjusted PnL,Gross Capital
0,2010-01-01,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,NaN,NaN,1,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0
1,2010-01-02,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,NaN,NaN,1,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0
2,2010-01-03,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,NaN,NaN,1,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0
3,2010-01-04,109.800003,0.000000,1117.699951,0.000000,0.000000,0.000000,NaN,NaN,1,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0
4,2010-01-05,109.699997,-0.000911,1118.099976,0.000358,-0.001269,0.001269,NaN,NaN,0,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0


In [3]:
# Filter columns to only keep those with "_Close", "_Size", or "_Cashflow"
filtered_columns = [col for col in data.columns if any(keyword in col for keyword in ["date","_Close","_dailyReturns", "_Size", "Live Trades"])]
df = data[filtered_columns]
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)
df

C:\Users\tee_m\AppData\Local\Temp\ipykernel_7424\3170661012.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


,date,GLD_Close,GLD_dailyReturns,GC=F_Close,GC=F_dailyReturns,GLD_Size,GC=F_Size,Live Trades
0,2010-07-02,118.489998,0.012389,1207.400024,0.000912,-8.0,10.0,Opened
1,2010-07-03,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live
2,2010-07-04,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live
3,2010-07-05,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live
4,2010-07-06,116.510002,-0.016710,1194.800049,-0.010436,8.0,-10.0,Closed
...,...,...,...,...,...,...,...,...
384,2025-01-28,255.179993,0.008656,2766.800049,0.010703,28.0,-10.0,Closed
385,2025-02-28,263.269989,-0.006266,2836.800049,-0.016093,-277.0,10.0,Opened
386,2025-03-01,263.269989,0.000000,2836.800049,0.000000,277.0,-10.0,Live
387,2025-03-02,263.269989,0.000000,2836.800049,0.000000,277.0,-10.0,Live


In [4]:
# Extract the part of the column name to the left of "_Close"
close_columns = [col for col in df.columns if "_Close" in col]
left_of_close = [col.split("_Close")[0] for col in close_columns]

# Display the extracted parts
print(left_of_close)
ticker1 = left_of_close[0]
ticker2 = left_of_close[1]

['GLD', 'GC=F']


In [5]:
dir = 0
for i in df.index:
    
    if df.loc[i ,"Live Trades"] == "Opened":
        if df.loc[i, ticker1+"_Size"] > 0:
            dir = 1
        else:
            dir = 0    
        
        while i < len(df) and df.loc[i, "Live Trades"] != "Closed":
            if dir==1:
                df.loc[i, ticker1+"_dir"] = "Long"
                df.loc[i, ticker2+"_dir"] = "Short"
            else:
                df.loc[i, ticker1+"_dir"] = "Short"
                df.loc[i, ticker2+"_dir"] = "Long"   
            i+=1     
            
df.ffill(inplace=True)
df     

C:\Users\tee_m\AppData\Local\Temp\ipykernel_7424\2649518441.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[i, ticker1+"_dir"] = "Short"
C:\Users\tee_m\AppData\Local\Temp\ipykernel_7424\2649518441.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[i, ticker2+"_dir"] = "Long"
C:\Users\tee_m\AppData\Local\Temp\ipykernel_7424\2649518441.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pyd

,date,GLD_Close,GLD_dailyReturns,GC=F_Close,GC=F_dailyReturns,GLD_Size,GC=F_Size,Live Trades,GLD_dir,GC=F_dir
0,2010-07-02,118.489998,0.012389,1207.400024,0.000912,-8.0,10.0,Opened,Short,Long
1,2010-07-03,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live,Short,Long
2,2010-07-04,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live,Short,Long
3,2010-07-05,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live,Short,Long
4,2010-07-06,116.510002,-0.016710,1194.800049,-0.010436,8.0,-10.0,Closed,Short,Long
...,...,...,...,...,...,...,...,...,...,...
384,2025-01-28,255.179993,0.008656,2766.800049,0.010703,28.0,-10.0,Closed,Short,Long
385,2025-02-28,263.269989,-0.006266,2836.800049,-0.016093,-277.0,10.0,Opened,Short,Long
386,2025-03-01,263.269989,0.000000,2836.800049,0.000000,277.0,-10.0,Live,Short,Long
387,2025-03-02,263.269989,0.000000,2836.800049,0.000000,277.0,-10.0,Live,Short,Long


In [6]:
for i in df.index:
    if df.loc[i, "Live Trades"] == "Opened":
        df.loc[i, ticker1+"_MTM"] = abs(df.loc[i, ticker1+"_Size"]*df.loc[i, ticker1+"_Close"])
        df.loc[i, ticker2+"_MTM"] = abs(df.loc[i, ticker2+"_Size"]*df.loc[i, ticker2+"_Close"])
        
        # print(i)
        i+=1
        # print(i)
        while i < len(df) and df.loc[i, "Live Trades"] != "Opened":
            
            if df.loc[i, ticker1+"_dir"] == "Long":
                df.loc[i, ticker1+"_MTM"] = df.loc[i-1, ticker1+"_MTM"] * (1+df.loc[i, ticker1+"_dailyReturns"])
                df.loc[i, ticker2+"_MTM"] = df.loc[i-1, ticker2+"_MTM"] * (1-df.loc[i, ticker2+"_dailyReturns"])

            else:
                df.loc[i, ticker1+"_MTM"] = df.loc[i-1, ticker1+"_MTM"] * (1-df.loc[i, ticker1+"_dailyReturns"])
                df.loc[i, ticker2+"_MTM"] = df.loc[i-1, ticker2+"_MTM"] * (1+df.loc[i, ticker2+"_dailyReturns"])
            
            i+=1
            
        i-=1
    
df

C:\Users\tee_m\AppData\Local\Temp\ipykernel_7424\335835895.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[i, ticker1+"_MTM"] = abs(df.loc[i, ticker1+"_Size"]*df.loc[i, ticker1+"_Close"])
C:\Users\tee_m\AppData\Local\Temp\ipykernel_7424\335835895.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[i, ticker2+"_MTM"] = abs(df.loc[i, ticker2+"_Size"]*df.loc[i, ticker2+"_Close"])


,date,GLD_Close,GLD_dailyReturns,GC=F_Close,GC=F_dailyReturns,GLD_Size,GC=F_Size,Live Trades,GLD_dir,GC=F_dir,GLD_MTM,GC=F_MTM
0,2010-07-02,118.489998,0.012389,1207.400024,0.000912,-8.0,10.0,Opened,Short,Long,947.919983,12074.000244
1,2010-07-03,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live,Short,Long,947.919983,12074.000244
2,2010-07-04,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live,Short,Long,947.919983,12074.000244
3,2010-07-05,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live,Short,Long,947.919983,12074.000244
4,2010-07-06,116.510002,-0.016710,1194.800049,-0.010436,8.0,-10.0,Closed,Short,Long,963.759949,11948.000488
...,...,...,...,...,...,...,...,...,...,...,...,...
384,2025-01-28,255.179993,0.008656,2766.800049,0.010703,28.0,-10.0,Closed,Short,Long,7028.942369,27668.000488
385,2025-02-28,263.269989,-0.006266,2836.800049,-0.016093,-277.0,10.0,Opened,Short,Long,72925.786957,28368.000488
386,2025-03-01,263.269989,0.000000,2836.800049,0.000000,277.0,-10.0,Live,Short,Long,72925.786957,28368.000488
387,2025-03-02,263.269989,0.000000,2836.800049,0.000000,277.0,-10.0,Live,Short,Long,72925.786957,28368.000488


In [7]:
df["TotalMTM"] = df[ticker1+"_MTM"] + df[ticker2+"_MTM"]
df["pct_change"] = df["TotalMTM"].pct_change()
df["pct_change"] = np.where(df["Live Trades"] == "Opened", 0 , df["pct_change"])
df["pnl"] = np.where(df["Live Trades"] != "Opened", df["TotalMTM"] - df["TotalMTM"].shift(1), 0)
# df["TotalMTM"].shift(1)
df

C:\Users\tee_m\AppData\Local\Temp\ipykernel_7424\2628036330.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TotalMTM"] = df[ticker1+"_MTM"] + df[ticker2+"_MTM"]
C:\Users\tee_m\AppData\Local\Temp\ipykernel_7424\2628036330.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["pct_change"] = df["TotalMTM"].pct_change()
C:\Users\tee_m\AppData\Local\Temp\ipykernel_7424\2628036330.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer

,date,GLD_Close,GLD_dailyReturns,GC=F_Close,GC=F_dailyReturns,GLD_Size,GC=F_Size,Live Trades,GLD_dir,GC=F_dir,GLD_MTM,GC=F_MTM,TotalMTM,pct_change,pnl
0,2010-07-02,118.489998,0.012389,1207.400024,0.000912,-8.0,10.0,Opened,Short,Long,947.919983,12074.000244,13021.920227,0.000000,0.000000
1,2010-07-03,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live,Short,Long,947.919983,12074.000244,13021.920227,0.000000,0.000000
2,2010-07-04,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live,Short,Long,947.919983,12074.000244,13021.920227,0.000000,0.000000
3,2010-07-05,118.489998,0.000000,1207.400024,0.000000,8.0,-10.0,Live,Short,Long,947.919983,12074.000244,13021.920227,0.000000,0.000000
4,2010-07-06,116.510002,-0.016710,1194.800049,-0.010436,8.0,-10.0,Closed,Short,Long,963.759949,11948.000488,12911.760437,-0.008460,-110.159790
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
384,2025-01-28,255.179993,0.008656,2766.800049,0.010703,28.0,-10.0,Closed,Short,Long,7028.942369,27668.000488,34696.942857,0.006720,231.623724
385,2025-02-28,263.269989,-0.006266,2836.800049,-0.016093,-277.0,10.0,Opened,Short,Long,72925.786957,28368.000488,101293.787445,0.000000,0.000000
386,2025-03-01,263.269989,0.000000,2836.800049,0.000000,277.0,-10.0,Live,Short,Long,72925.786957,28368.000488,101293.787445,0.000000,0.000000
387,2025-03-02,263.269989,0.000000,2836.800049,0.000000,277.0,-10.0,Live,Short,Long,72925.786957,28368.000488,101293.787445,0.000000,0.000000


In [8]:
import calculate_stats

In [9]:
df_index = df[["date","pct_change"]]
df_index
df_index.to_csv("test.csv")
df_index

,date,pct_change
0,2010-07-02,0.000000
1,2010-07-03,0.000000
2,2010-07-04,0.000000
3,2010-07-05,0.000000
4,2010-07-06,-0.008460
...,...,...
384,2025-01-28,0.006720
385,2025-02-28,0.000000
386,2025-03-01,0.000000
387,2025-03-02,0.000000


In [10]:
calculate_stats.get_sharpe_ratio(df_index)

0.6781101049447967